In [ ]:
import os, glob
import numpy as np
import pandas as pd
import cv2
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import StratifiedGroupKFold

INNER = "/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2"
OUT_DIR = "/kaggle/working"
TYPES = ["RealVideo-RealAudio", "RealVideo-FakeAudio",
         "FakeVideo-RealAudio", "FakeVideo-FakeAudio"]
MIN_FRAMES = 8
MIN_SIZE = 10_000   # bytes
MIN_REAL_PER_SPLIT = 50

assert os.path.isdir(INNER), "Dataset path is wrong. Check Add Input and the path."

In [ ]:
rows = []
for t in TYPES:
    for p in glob.glob(f"{INNER}/{t}/**/*.mp4", recursive=True):
        parts = os.path.relpath(p, INNER).split(os.sep)   # type/race/gender/speaker/file
        rows.append(dict(path=p, type=parts[0], race=parts[1], gender=parts[2],
                         speaker=parts[3], filename=parts[-1], depth=len(parts)))

df = pd.DataFrame(rows)
print("videos found:", len(df))
print("path depth counts (should all be 5):")
print(df["depth"].value_counts())
print(df["type"].value_counts())
df.head()

In [ ]:
df["label"] = (df["type"] != "RealVideo-RealAudio").astype(int)      # 1 = fake
df["video_fake"] = df["type"].str.startswith("FakeVideo").astype(int)
df["audio_fake"] = df["type"].str.endswith("FakeAudio").astype(int)

print(df["label"].value_counts())
print(df.groupby("type")[["label", "video_fake", "audio_fake"]].mean())

In [ ]:
print("duplicate paths:", df.duplicated("path").sum())
df = df.drop_duplicates("path").reset_index(drop=True)

df["size"] = df["path"].map(os.path.getsize)
print("tiny files:", (df["size"] < MIN_SIZE).sum())

def check(p):
    try:
        cap = cv2.VideoCapture(p)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        ok, _ = cap.read()
        cap.release()
        return bool(ok), n, fps
    except Exception:
        return False, 0, 0.0

with ThreadPoolExecutor(max_workers=8) as ex:
    res = list(ex.map(check, df["path"]))

df["readable"] = [r[0] for r in res]
df["n_frames"] = [r[1] for r in res]
df["fps"] = [r[2] for r in res]
df["duration"] = df["n_frames"] / df["fps"].replace(0, np.nan)

print("unreadable:", (~df["readable"]).sum())
print("fewer than", MIN_FRAMES, "frames:", (df["n_frames"] < MIN_FRAMES).sum())
print(df[["n_frames", "fps", "duration"]].describe())

In [ ]:
before = len(df)
df = df[df["readable"] & (df["n_frames"] >= MIN_FRAMES) & (df["size"] >= MIN_SIZE)]
df = df.reset_index(drop=True)
print(f"kept {len(df)} / {before}")
print(df["type"].value_counts())
print(df["label"].value_counts())

In [ ]:
def make_split(df, seed):
    sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)
    folds = list(sgkf.split(df, df["label"], groups=df["speaker"]))
    split = np.array(["train"] * len(df), dtype=object)
    split[folds[0][1]] = "val"
    split[folds[1][1]] = "test"
    return split

chosen = None
for seed in range(100):
    split = make_split(df, seed)
    real = df["label"].eq(0)
    n_val = ((split == "val") & real).sum()
    n_test = ((split == "test") & real).sum()
    n_train = ((split == "train") & real).sum()
    if n_val >= MIN_REAL_PER_SPLIT and n_test >= MIN_REAL_PER_SPLIT and n_train >= 250:
        chosen = seed
        break

assert chosen is not None, "No seed gave enough real videos per split. Lower MIN_REAL_PER_SPLIT."
df["split"] = make_split(df, chosen)
print("chosen seed:", chosen)

# no speaker in two splits
assert (df.groupby("speaker")["split"].nunique() == 1).all(), "speaker leakage!"

print(df.groupby(["split", "label"]).size().unstack())
print(df.groupby(["split", "type"]).size().unstack())
print("speakers per split:", df.groupby("split")["speaker"].nunique().to_dict())

In [ ]:
cols = ["path", "filename", "type", "race", "gender", "speaker",
        "label", "video_fake", "audio_fake", "n_frames", "fps", "duration", "split"]
os.makedirs(OUT_DIR, exist_ok=True)
df[cols].to_csv(f"{OUT_DIR}/master.csv", index=False)

check_df = pd.read_csv(f"{OUT_DIR}/master.csv")
print(check_df.shape)
check_df.head()